# Лабораторна робота 3 — Градієнтний спуск для множинної регресії

**Набір даних:** `kc_house_data.csv`  
**Обмеження:** scikit-learn-регресія **не дозволена** для базових завдань.

## Налаштування

In [ ]:
import sys
!{sys.executable} -m pip install numpy pandas matplotlib --quiet


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt
from sklearn.model_selection import train_test_split

%matplotlib inline


## Теоретичне підґрунтя

Градієнт RSS відносно ваги *wᵢ*:
```
gradient_i = 2 · dot(errors, feature_i)     де  errors = Xw − y
```
Зупинка відбувається, коли ‖gradient‖₂ < `tolerance`.

---
## Завдання 1 — Підготовка даних та `get_numpy_data()`

Завантажте `kc_house_data.csv`. Розбийте **20 % навчання / 80 % тест** (`random_state=0`).

Реалізуйте `get_numpy_data(dataframe, features, output)`, яка:
1. Додає стовпець з одиницями ліворуч від матриці ознак (для вільного члена).
2. Повертає `(feature_matrix, output_array)` як масиви NumPy.

In [ ]:
sales = pd.read_csv('kc_house_data.csv')
train_data, test_data = train_test_split(sales, test_size=0.8, random_state=0)


In [ ]:
def get_numpy_data(dataframe, features, output):
    """
    Будує матрицю ознак NumPy (зі стовпцем вільного члена) та вектор виходу.

    Параметри
    ----------
    dataframe : pd.DataFrame
    features  : список str   назви стовпців-ознак
    output    : str          назва стовпця цільової змінної

    Повертає
    -------
    feature_matrix : np.ndarray, shape (n, len(features)+1)
    output_array   : np.ndarray, shape (n,)
    """
    # Add a constant column of 1s (intercept)
    dataframe = dataframe.copy()
    dataframe['constant'] = 1.0

    # Select feature columns and convert to NumPy
    feature_matrix = dataframe[['constant'] + features].to_numpy(dtype=float)

    output_array = dataframe[output].to_numpy(dtype=float)
    return feature_matrix, output_array


### Перевірка

In [ ]:
example_features, example_output = get_numpy_data(sales, ['sqft_living'], 'price')
print('Перший рядок матриці ознак:', example_features[0, :])  # має бути [1.0, <sqft>]
print('Перше значення виходу:    ', example_output[0])


Перший рядок матриці ознак: [1.00e+00 1.18e+03]
Перше значення виходу:     221900.0


---
## Завдання 2 — Реалізація `predict_output()`

Використовуйте **один виклик `np.dot`** — без явних циклів.

In [ ]:
def predict_output(feature_matrix, weights):
    """
    Обчислює передбачення як скалярний добуток кожного рядка з вагами.

    Повертає
    -------
    predictions : np.ndarray, shape (n,)
    """
    predictions = np.dot(feature_matrix, weights)

    return predictions


### Перевірка — перші два передбачення мають бути ≈1181 і ≈2571

In [ ]:
my_weights = np.array([1.0, 1.0])
test_preds = predict_output(example_features, my_weights)
print(f'передбачення[0]: {test_preds[0]:.1f}  (очікується ≈1181)')
print(f'передбачення[1]: {test_preds[1]:.1f}  (очікується ≈2571)')


передбачення[0]: 1181.0  (очікується ≈1181)
передбачення[1]: 2571.0  (очікується ≈2571)


---
## Завдання 3 — Реалізація `feature_derivative()`

Похідна RSS відносно однієї ваги — це `2 · dot(errors, feature)`. Реалізуйте як один вираз NumPy.

In [ ]:
def feature_derivative(errors, feature):
    """Повертає похідну RSS відносно ваги для даної ознаки."""
    return 2 * np.dot(errors, feature)


### Перевірка — похідна відносно константної ознаки має дорівнювати `-2 · sum(prices)`

In [ ]:
zero_weights = np.array([0.0, 0.0])
zero_preds   = predict_output(example_features, zero_weights)
errors       = zero_preds - example_output              # = -prices when weights=0

deriv_constant = feature_derivative(errors, example_features[:, 0])
expected       = -2 * np.sum(example_output)
print(f'Обчислено: {deriv_constant:.2e}')
print(f'Очікується: {expected:.2e}')
print(f'Збіг: {np.isclose(deriv_constant, expected)}')


Обчислено: -2.33e+10
Очікується: -2.33e+10
Збіг: True


---
## Завдання 4 — Градієнтний спуск

Реалізуйте `regression_gradient_descent(feature_matrix, output, initial_weights, step_size, tolerance)`. Функція оновлює всі ваги одночасно на кожній ітерації та повертає їх, коли ‖gradient‖₂ < `tolerance`.

Запустіть з такими параметрами на **навчальних** даних:
```
features        = ['sqft_living']
initial_weights = [-47000., 1.]
step_size       = 7e-12
tolerance       = 2.5e7
```
Вкажіть навчені ваги. Яка передбачувана ціна першого будинку в **тестовій** вибірці? Обчисліть RSS на всій тестовій вибірці.

In [ ]:
def regression_gradient_descent(feature_matrix, output,
                                initial_weights, step_size, tolerance):
    """
    Мінімізує RSS за допомогою пакетного градієнтного спуску.

    Повертає
    -------
    weights : np.ndarray — навчені ваги
    """
    weights   = np.array(initial_weights, dtype=float)
    converged = False

    while not converged:
        # 1. Compute predictions
        predictions = predict_output(feature_matrix, weights)

        # 2. Compute errors
        errors = predictions - output

        gradient_sum_squares = 0.0
        for i in range(len(weights)):
            # 3. Derivative for weight i
            derivative = feature_derivative(errors, feature_matrix[:, i])

            # 4. Accumulate squared gradient
            gradient_sum_squares += derivative ** 2

            # 5. Update weight
            weights[i] -= step_size * derivative

        # 6. Check convergence
        gradient_magnitude = sqrt(gradient_sum_squares)
        if gradient_magnitude < tolerance:
            converged = True

    return weights


### Запуск Моделі 1 — одна ознака

In [ ]:
simple_feature_matrix, output = get_numpy_data(train_data, ['sqft_living'], 'price')

simple_weights = regression_gradient_descent(
    simple_feature_matrix, output,
    initial_weights=[-47000., 1.],
    step_size=7e-12,
    tolerance=2.5e7
)
print('Навчені ваги:', simple_weights)


Навчені ваги: [-46999.88018846    280.51005013]


### Оцінка Моделі 1 на тестовій вибірці

In [ ]:
test_feature_matrix, test_output = get_numpy_data(test_data, ['sqft_living'], 'price')

# Predicted price for the first test house
test_predictions = predict_output(test_feature_matrix, simple_weights)
print(f'Передбачена ціна (будинок 0): ${test_predictions[0]:,.0f}')
print(f'Реальна ціна    (будинок 0): ${test_output[0]:,.0f}')

# RSS on the full test set
rss_model1 = np.sum((test_predictions - test_output) ** 2)
print(f'Тестова RSS Моделі 1: {rss_model1:.3e}')


Передбачена ціна (будинок 0): $354,129
Реальна ціна    (будинок 0): $297,000
Тестова RSS Моделі 1: 1.204e+15


---
## ✨ Бонус — Додайте другу ознаку

Повторіть градієнтний спуск з `['sqft_living', 'sqft_living15']` та параметрами:
```
initial_weights = [-100000., 1., 1.]
step_size       = 4e-12
tolerance       = 1e9
```
Обчисліть RSS на тестовій вибірці та порівняйте з моделлю з однією ознакою. Порівняйте передбачену ціну першого тестового будинку з обох моделей з реальною ціною.

In [ ]:
# Бонус — модель з двома ознаками
# Параметри: initial_weights=[-100000., 1., 1.], step_size=4e-12, tolerance=1e9
train_feature_matrix, train_output = get_numpy_data(
    train_data,
    ['sqft_living', 'sqft_living15'],
    'price'
)

two_feature_weights = regression_gradient_descent(
    train_feature_matrix,
    train_output,
    initial_weights=[-100000., 1., 1.],
    step_size=4e-12,
    tolerance=1e9
)

print('Навчені ваги:', two_feature_weights)

test_feature_matrix, test_output = get_numpy_data(
    test_data,
    ['sqft_living', 'sqft_living15'],
    'price'
)

test_predictions_2 = predict_output(
    test_feature_matrix,
    two_feature_weights
)

rss_model2 = np.sum((test_predictions_2 - test_output) ** 2)

print(f'Передбачена ціна (Модель 2): ${test_predictions_2[0]:,.0f}')
print(f'Реальна ціна:                ${test_output[0]:,.0f}')
print(f'Тестова RSS Моделі 2: {rss_model2:.3e}')

print(f'Тестова RSS Моделі 1: {rss_model1:.3e}')

Навчені ваги: [-9.99998826e+04  2.43610628e+02  6.54855462e+01]
Передбачена ціна (Модель 2): $342,008
Реальна ціна:                $297,000
Тестова RSS Моделі 2: 1.186e+15
Тестова RSS Моделі 1: 1.204e+15
